# ByteCore-DualMemory — 4-Hour Pre-Training (T4)

**Phase 24 pipeline, end to end on a free Colab T4.**

- **Model:** 8-layer, 1024-dim byte Transformer (265,618,710 params) with a
  **Gated Skill Adapter** (1024 ↔ 8192 GLU bridge) in every layer, a
  confidence-headed adaptive refinement loop (≤4 passes, early break at 0.9),
  and frozen multimodal encoders projecting into the 8192-dim skill bank.
- **Data mix (streaming):** 40% code (`the-stack` Python/JS) · 30% language
  (`GigaVerbo` PT-BR + `fineweb` EN) · 20% multimodal (COYO images + LibriSpeech
  audio, bidirectional byte prediction) · 10% logic (`open-web-math`).
- **VRAM:** bf16 on Ampere+; **fp16 + GradScaler on T4** (sm_75 cannot bf16).
  Gradient checkpointing, `batch_size=4`, `grad_accum=8`, measured VRAM probe
  with automatic batch shrink.
- **Checkpoints:** atomic, every **30 minutes**, pushed to
  `gabrielbosse9/airesearch` on the Hugging Face Hub. Re-running this notebook
  resumes from the latest checkpoint automatically.

> Runtime: **Runtime → Change runtime type → T4 GPU** before starting.
> Add your Hugging Face token as the Colab secret `HF_TOKEN` (write access to
> `gabrielbosse9/airesearch`), or enter it interactively in the auth cell.
> Total wall clock ≈ 4 h: ~10 min setup + 220 min training + eval/upload.

In [ ]:
# 1) GPU check — fail fast if the runtime is not a GPU.
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} (sm_{major}{minor}), {total_gb:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    print("T4 detected -> trainer resolves to float16 + GradScaler (required fallback)")

In [ ]:
# 2) Dependencies. torch/torchvision/torchaudio ship with Colab; add the rest.
import subprocess
import sys

packages = ["datasets>=3,<6", "huggingface_hub>=0.25", "pyyaml", "ddgs",
            "nbformat>=5", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("dependencies ready")

In [ ]:
# 3) Repository. Uses the merged Phase 24 main branch.
import os
import subprocess

REPO_URL = "https://github.com/gabrielbosse1/airesearch.git"
REPO_DIR = "/content/airesearch"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

In [ ]:
# 4) Hugging Face auth (checkpoints every 30 min go to gabrielbosse9/airesearch).
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import login
if token:
    login(token=token, add_to_git_credential=False)
else:
    login()  # interactive widget fallback
print("Hugging Face auth ready")

In [ ]:
# 5) Configuration + exact parameter count (meta device: zero memory cost).
import torch
import yaml

from models.foundation import CausalByteTransformer, FoundationConfig

with open("configs/t4_scaled.yaml", "r", encoding="utf8") as handle:
    config = yaml.safe_load(handle)
cfg = FoundationConfig.from_mapping(config["model"])
print("layers:", cfg.n_layers, "| d_model:", cfg.d_model, "| context:", cfg.max_seq_len)
print("memory_dim:", cfg.memory_dim, "| refinement:", cfg.refinement_steps,
      "| confidence threshold:", cfg.confidence_threshold)
with torch.device("meta"):
    probe = CausalByteTransformer(cfg)
print(f"parameters: {probe.parameters_count:,}")
print("data weights:", config["data"]["weights"])

## Training

One unified script: `python -m training.train`. It streams the 40/30/20/10 mix,
builds **bidirectional multimodal batches** (caption bytes ⇄ media codec bytes),
and optimises causal-LM cross-entropy + the auxiliary confidence BCE.

- Checkpoints: `runs/checkpoints/checkpoint_latest.pt`, every 30 min + every 500
  steps, uploaded to `gabrielbosse9/airesearch/checkpoints/checkpoint_latest.pt`.
- The VRAM probe measures a real forward+backward before step 0 and shrinks the
  micro-batch 4 → 2 → 1 if the peak exceeds 12 GB.
- If Colab disconnects, just re-run the notebook: training resumes from the
  latest local or Hub checkpoint.

In [ ]:
# 6) The 4-hour run (220-minute training budget inside the ~4 h session).
import subprocess
import sys
import time

HF_REPO = "gabrielbosse9/airesearch"
cmd = [sys.executable, "-u", "-m", "training.train",
       "--config", "configs/t4_scaled.yaml",
       "--hf-repo", HF_REPO,
       "--max-minutes", "220",
       "--checkpoint-minutes", "30"]
print("launching:", " ".join(cmd), flush=True)
start = time.time()
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    exit_code = proc.wait()
print(f"training exit code {exit_code} after {(time.time() - start) / 60:.1f} min")
assert exit_code == 0, "training failed -- inspect the log above"


In [ ]:
# 7) Post-training sanity: generation, adaptive loop stats, true tooling demo.
import torch

from models.foundation import (CausalByteTransformer, FoundationConfig,
                               decode_ids, encode_bytes)
from training.train import make_search_fn

ckpt = torch.load("runs/checkpoints/checkpoint_latest.pt", map_location="cuda",
                  weights_only=False)
model = CausalByteTransformer(FoundationConfig(**ckpt["config"])).cuda()
model.load_state_dict(ckpt["model"])
model.eval()
print("resumed checkpoint at step", ckpt["step"])

prompt = encode_bytes(b"The capital of Brazil is").cuda()
sample = model.generate(prompt, max_new_bytes=64)
print("sample:", decode_ids(sample))

_, info = model(prompt.unsqueeze(0), return_info=True)
print("adaptive loop:", {"steps_run": info["steps_run"], "confidence": info["confidence"]})

# [TOOL_SEARCH] token ids trigger a real sandboxed search (mock backend here;
# pass make_search_fn("ddgs") for live DuckDuckGo on this networked runtime).
ids, stats = model.generate_with_tools(encode_bytes(b"search: ").cuda(),
                                       make_search_fn("mock"), max_new_bytes=96)
print("tool demo:", stats)
print("transcript:", decode_ids(ids, keep_controls=True)[:300])

In [ ]:
# 8) Final artifacts to the Hub (the trainer already pushed the latest ckpt).
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
artifacts = [
    ("runs/checkpoints/checkpoint_latest.pt", "checkpoints/checkpoint_final.pt"),
    ("configs/t4_scaled.yaml", "configs/t4_scaled.yaml"),
    ("runs/metrics.jsonl", "logs/metrics.jsonl"),
]
for local_path, repo_path in artifacts:
    try:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path,
                        repo_id=HF_REPO, repo_type="model")
        print("uploaded:", repo_path)
    except Exception as exc:
        print("skipped:", local_path, "->", exc)
print("done -- model, config and training log are on the Hub")

## Honest expectations

- 220 minutes on a T4 is a **small token budget** for a 265M-parameter model
  (order 10⁸ tokens). Expect coherent byte-level structure and falling loss,
  not a finished assistant. Continue pre-training by re-running (auto-resume).
- COYO images are fetched by URL; dead links are skipped and the mixture
  renormalises. The embedded MSCOCO-5k and LibriSpeech fallbacks keep the
  multimodal fraction alive when COYO fails.
- The T4 runs **fp16 + GradScaler**, not bf16 — this is resolved and logged at
  runtime, never assumed.
- `[EDIT]` tokens are ordinary vocabulary learned by cross-entropy; only
  `[TOOL_SEARCH]` ids execute anything, inside a resource-limited subprocess.